<center><font color='steelblue'> <font size = "5,5">CORPORACIÓN UNIVERSITARIA MINUTO DE DIOS </font></center><br>
<center><font color='steelblue'> <font size = "5">FACULTAD DE INGENIERÍA</font></center><br>
<center><font color='steelblue'> <font size = "4">PROGRAMA INGENIERÍA DE SISTEMAS</font></center><br>
<center><font color='steelblue'> <font size = "3">CURSO BASES DE DATOS MASIVAS</font></center><br>
<center><font color="yellow" size = "4" face = "small fonts">Proyecto Modular - Mojo</font></center>

<center><font color="olive" size = "4" face = "small fonts">DATOS DE LOS PARTICIPANTES DEL GRUPO</font></center>

<font color="yellow" size = "4" face = "small fonts">NRC: </font></center><br>
<font color="yellow" size = "4" face = "small fonts">Nombres: </font></center><br>
<font color="yellow" size = "4" face = "small fonts">ID:</font></center><br>

<font color="yellow" size = "4" face = "small fonts">El taller retoma algunas de las instrucciones utilizadas a través del curso de Bases de Datos Masivas, tener en cuenta seguir los pasos requeridos.</font></center><br>

<font color="yellow" size = "4" face = "small fonts">Al Cargar los archivos dispuestos en la carpeta Poryecto final BDM se debe utilizar sentencias que no fijen el path, este debe se dinámico en caso de que se creen nuevos paquetes</font></center><br>

<font color="red" size = "4" face = "small fonts">El ejercicio consta de seguir el cuaderno y  realizar las tareas solicitadas, Utilizar y citar la documentación propia de cada una de las herramientas utilizadas para realizar el tratamiento a la data.</font></center><br>

<font color="yellow" size = "4" face = "small fonts">1. Configuración e importe de las librerias.</font></center><br>

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import requests
import re
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import subprocess

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'data'))
SQLITE_PATH = os.path.join(DATA_DIR, 'almacen.sqlite')

print(f'Directorio notebook : {NOTEBOOK_DIR}')
print(f'Directorio data     : {DATA_DIR}')
print(f'Ruta SQLite         : {SQLITE_PATH}')

<font color="yellow" size = "4" face = "small fonts">2. Configure y levante los contenedores necesarios para la actividad, recuerde que la arquitectura respeta a 3 contenedores con una distribucion de DBMongo en cada una de ellas, un contenedor que tiene configurado Mojolicious con todas sus dependencias junto con una base de datos Relacional la cual sera escogida a su gusto.</font></center><br>

In [ ]:
os.chdir(DATA_DIR)
print(f'Directorio de trabajo actual: {os.getcwd()}')

resultado = os.system('docker-compose up -d --build')
if resultado == 0:
    print('Contenedores levantados correctamente.')
else:
    print('Error al levantar los contenedores. Verifica que Docker Desktop esté corriendo.')

resultado = subprocess.run(
    'docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}"',
    shell=True, capture_output=True, text=True
)
print(resultado.stdout)

<font color="yellow" size = "4" face = "small fonts">3. Una vez tenga el contenedor arriba cargue los datos .Json, uno por cada contenedor DBMongo.</font></center><br>

<font color="yellow" size = "4" face = "small fonts">Tenga en cuenta que debe tener la replica de los formatos `JSon` en cada contenedor</font></center><br>

In [ ]:
JSON_PATH_CONTAINER = '/data'

cmd_personas = (
    f'docker exec mongodb_personas mongoimport '
    f'--db almacen --collection personas --file {JSON_PATH_CONTAINER}/personas.json '
    f'--jsonArray --drop'
)
resultado_personas = os.system(cmd_personas)
print('personas.json importado' if resultado_personas == 0 else 'Error importando personas.json')



cmd_articulos = (
    f'docker exec mongodb_articulos mongoimport '
    f'--db almacen --collection articulos --file {JSON_PATH_CONTAINER}/articulos.json '
    f'--jsonArray --drop'
)
resultado_articulos = os.system(cmd_articulos)
print('articulos.json importado' if resultado_articulos == 0 else 'Error importando articulos.json')




cmd_ventas = (
    f'docker exec mongodb_ventas mongoimport '
    f'--db almacen --collection ventas --file {JSON_PATH_CONTAINER}/ventas.json '
    f'--jsonArray --drop'
)
resultado_ventas = os.system(cmd_ventas)
print('ventas.json importado' if resultado_ventas == 0 else 'Error importando ventas.json')




<font color="yellow" size = "4" face = "small fonts">Se debe colocar el puerto expuesto del contenedor `Mojo`, para llamar la función y cargar los datos</font></center><br>

In [ ]:
MOJO_URL = 'http://localhost:8080'

<font color="yellow" size = "4" face = "small fonts">4. Muestre los datos que fueron almacenados en cada una de las distribuciones de BDMongo.</font></center><br>

In [ ]:
for nombre, contenedor, coleccion in [
    ('personas', 'mongodb_personas', 'personas'),
    ('articulos', 'mongodb_articulos', 'articulos'),
    ('ventas', 'mongodb_ventas', 'ventas')
]:
    print(f'--- Conteo: {nombre} ---')
    resultado = subprocess.run(
        f'docker exec {contenedor} mongosh --quiet --eval "db.getSiblingDB(\'almacen\').{coleccion}.countDocuments()"',
        shell=True, capture_output=True, text=True
    )
    print(resultado.stdout or resultado.stderr)


<font color="yellow" size = "4" face = "small fonts">Consultar a tráves del puerto expuesto la data que se encuentran en las colecciones personas, articulos y ventas de DBMongo</font></center><br>

In [ ]:
resultado = subprocess.run(f'curl -s {MOJO_URL}/mongo/personas', shell=True, capture_output=True, text=True)
print('--- Personas ---')
print(resultado.stdout)

In [ ]:
resultado = subprocess.run(f'curl -s {MOJO_URL}/mongo/articulos', shell=True, capture_output=True, text=True)
print('--- Articulos ---')
print(resultado.stdout)

In [ ]:
resultado = subprocess.run(f'curl -s {MOJO_URL}/mongo/ventas', shell=True, capture_output=True, text=True)
print('--- Ventas ---')
print(resultado.stdout)

<font color="yellow" size = "4" face = "small fonts">5. Verifique la estructura y tipo de los datos almacenados en las distribuciones DBMongo, genere una ETL para almacenar cada una de las distribuciones en la base de datos Relacional que configuro con anterioridad, recuerde que cada distribucion debe ir en una tabla relacional respetando la integridad referencial.</font></center><br>

In [ ]:
def limpiar_valor_numerico(valor):
    if valor is None:
        return None
    solo_digitos = re.sub(r'[^0-9]', '', str(valor))
    return int(solo_digitos) if solo_digitos else None


def filtrar_y_limpiar_articulos(articulos):
    limpios = []
    for doc in articulos:
        doc.pop('_id', None)
        id_art   = limpiar_valor_numerico(doc.get('idArticulo'))
        cantidad = limpiar_valor_numerico(doc.get('cantidadArticulo'))
        if id_art is None or cantidad is None:
            continue
        limpios.append({
            'idArticulo'      : id_art,
            'nombreArticulo'  : str(doc.get('nombreArticulo', '')).strip(),
            'precioArticulo'  : float(doc.get('precioArticulo', 0)),
            'cantidadArticulo': cantidad
        })
    return limpios


response_art = requests.get(f'{MOJO_URL}/mongo/articulos')
articulos_raw = response_art.json()

print(f'Documentos recibidos (artículos): {len(articulos_raw)}')
print('Estructura del primer documento:')
if articulos_raw:
    for campo, valor in articulos_raw[0].items():
        print(f'  {campo}: {type(valor).__name__} → {valor}')

articulos_limpios = filtrar_y_limpiar_articulos(articulos_raw)
print(f'\nDocumentos válidos tras ETL: {len(articulos_limpios)}')
print(f'Descartados por datos no numéricos: {len(articulos_raw) - len(articulos_limpios)}')

conexion = sqlite3.connect(SQLITE_PATH)
cursor = conexion.cursor()
cursor.execute('PRAGMA foreign_keys = ON')
cursor.execute('''
    CREATE TABLE IF NOT EXISTS articulos (
        idArticulo       INTEGER PRIMARY KEY,
        nombreArticulo   TEXT    NOT NULL,
        precioArticulo   REAL    NOT NULL,
        cantidadArticulo INTEGER NOT NULL
    )
''')
cursor.execute('DELETE FROM articulos')
cursor.executemany(
    'INSERT OR REPLACE INTO articulos VALUES (:idArticulo, :nombreArticulo, :precioArticulo, :cantidadArticulo)',
    articulos_limpios
)
conexion.commit()
conexion.close()
print('\nTabla articulos cargada en SQLite.')

In [ ]:
response = requests.get(f'{MOJO_URL}/mongo/personas')
personas_raw = response.json()

print(f'Documentos recibidos (personas): {len(personas_raw)}')
print('Estructura del primer documento:')
if personas_raw:
    for campo, valor in personas_raw[0].items():
        print(f'  {campo}: {type(valor).__name__} → {valor}')

personas_limpias = []
for doc in personas_raw:
    doc.pop('_id', None)
    num_doc  = limpiar_valor_numerico(doc.get('numeroDocumento'))
    telefono = limpiar_valor_numerico(doc.get('telefono'))
    if num_doc is None or telefono is None:
        continue
    personas_limpias.append({
        'numeroDocumento' : num_doc,
        'nombres'         : str(doc.get('nombres', '')).strip(),
        'primerApellido'  : str(doc.get('primerApellido', '')).strip(),
        'segundoApellido' : str(doc.get('segundoApellido', '')).strip(),
        'fechaNacimiento' : str(doc.get('fechaNacimiento', '')).strip(),
        'telefono'        : telefono,
        'direccion'       : str(doc.get('direccion', '')).strip(),
        'email'           : str(doc.get('email', '')).strip()
    })

print(f'\nDocumentos válidos tras ETL: {len(personas_limpias)}')
print(f'Descartados por datos no numéricos: {len(personas_raw) - len(personas_limpias)}')



conexion = sqlite3.connect(SQLITE_PATH)
cursor = conexion.cursor()
cursor.execute('PRAGMA foreign_keys = ON')
cursor.execute('''
    CREATE TABLE IF NOT EXISTS personas (
        numeroDocumento  INTEGER PRIMARY KEY,
        nombres          TEXT NOT NULL,
        primerApellido   TEXT NOT NULL,
        segundoApellido  TEXT,
        fechaNacimiento  TEXT,
        telefono         INTEGER,
        direccion        TEXT,
        email            TEXT
    )
''')
cursor.execute('DELETE FROM personas')
cursor.executemany(
    '''INSERT OR REPLACE INTO personas VALUES
       (:numeroDocumento, :nombres, :primerApellido, :segundoApellido,
        :fechaNacimiento, :telefono, :direccion, :email)''',
    personas_limpias
)
conexion.commit()
conexion.close()
print('\nTabla personas cargada en SQLite.')

In [ ]:
response = requests.get(f'{MOJO_URL}/mongo/ventas')
ventas_raw = response.json()

print(f'Documentos recibidos (ventas): {len(ventas_raw)}')
print('Estructura del primer documento:')
if ventas_raw:
    for campo, valor in ventas_raw[0].items():
        print(f'  {campo}: {type(valor).__name__} → {valor}')

ventas_limpias = []
for doc in ventas_raw:
    doc.pop('_id', None)
    id_venta     = limpiar_valor_numerico(doc.get('idVenta'))
    id_comprador = limpiar_valor_numerico(doc.get('idComprador'))
    id_articulo  = limpiar_valor_numerico(doc.get('idArticulo'))
    cantidad     = limpiar_valor_numerico(doc.get('cantidadProductos'))
    if id_comprador is None or id_articulo is None or cantidad is None:
        continue
    ventas_limpias.append({
        'idVenta'          : id_venta,
        'idComprador'      : id_comprador,
        'idArticulo'       : id_articulo,
        'cantidadProductos': cantidad,
        'precioTotal'      : float(doc.get('precioTotal', 0))
    })

print(f'\nDocumentos válidos tras ETL: {len(ventas_limpias)}')
print(f'Descartados por datos no numéricos: {len(ventas_raw) - len(ventas_limpias)}')



conexion = sqlite3.connect(SQLITE_PATH)
ids_personas  = set(pd.read_sql_query('SELECT numeroDocumento FROM personas', conexion)['numeroDocumento'])
ids_articulos = set(pd.read_sql_query('SELECT idArticulo FROM articulos', conexion)['idArticulo'])

ventas_validas = [
    v for v in ventas_limpias
    if v['idComprador'] in ids_personas and v['idArticulo'] in ids_articulos
]

print(f'Ventas válidas FK : {len(ventas_validas)}')
print(f'Descartadas por FK: {len(ventas_limpias) - len(ventas_validas)}')

cursor = conexion.cursor()
cursor.execute('PRAGMA foreign_keys = ON')
cursor.execute('''
    CREATE TABLE IF NOT EXISTS ventas (
        idVenta           INTEGER PRIMARY KEY,
        idComprador       INTEGER NOT NULL,
        idArticulo        INTEGER NOT NULL,
        cantidadProductos INTEGER NOT NULL,
        precioTotal       REAL    NOT NULL,
        FOREIGN KEY (idComprador) REFERENCES personas(numeroDocumento),
        FOREIGN KEY (idArticulo)  REFERENCES articulos(idArticulo)
    )
''')
cursor.execute('DELETE FROM ventas')
cursor.executemany(
    '''INSERT OR REPLACE INTO ventas VALUES
       (:idVenta, :idComprador, :idArticulo, :cantidadProductos, :precioTotal)''',
    ventas_validas
)
conexion.commit()
conexion.close()
print('\nTabla ventas cargada en SQLite.')

<font color="yellow" size = "4" face = "small fonts">6. Muestre la data almacenada en la base de datos Relacional que esta usando, tabla por tabla.</font></center><br>

In [ ]:

conexion = sqlite3.connect(SQLITE_PATH)


consulta = 'SELECT * FROM personas'
df = pd.read_sql_query(consulta, conexion)


print(f'Tabla personas — {len(df)} registros')
display(df)


conexion.close()

In [ ]:

conexion = sqlite3.connect(SQLITE_PATH)

consulta = 'SELECT * FROM articulos'
df = pd.read_sql_query(consulta, conexion)

print(f'Tabla articulos — {len(df)} registros')
display(df)

conexion.close()

In [ ]:
conexion = sqlite3.connect(SQLITE_PATH)

consulta = 'SELECT * FROM ventas'
df = pd.read_sql_query(consulta, conexion)

print(f'Tabla ventas — {len(df)} registros')
display(df)

conexion.close()

<font color="yellow" size = "4" face = "small fonts">7. Grafique los 5 articulos mas vendidos.</font></center><br>

In [ ]:
conexion = sqlite3.connect(SQLITE_PATH)

consulta = """
    SELECT a.nombreArticulo, SUM(v.cantidadProductos) AS total_vendido
    FROM ventas v
    JOIN articulos a ON v.idArticulo = a.idArticulo
    GROUP BY v.idArticulo
    ORDER BY total_vendido DESC
    LIMIT 5
"""
df = pd.read_sql_query(consulta, conexion)
conexion.close()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(df['nombreArticulo'], df['total_vendido'], color='steelblue')
ax.bar_label(bars, padding=4)
ax.set_xlabel('Unidades vendidas')
ax.set_title('Top 5 artículos más vendidos')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


<font color="yellow" size = "4" face = "small fonts">8. Grafique los 5 compradores que han realizado más compras.</font></center><br>

In [ ]:
conexion = sqlite3.connect(SQLITE_PATH)

consulta = """
    SELECT p.nombres || ' ' || p.primerApellido AS comprador,
           COUNT(v.idVenta) AS num_compras
    FROM ventas v
    JOIN personas p ON v.idComprador = p.numeroDocumento
    GROUP BY v.idComprador
    ORDER BY num_compras DESC
    LIMIT 5
"""
df = pd.read_sql_query(consulta, conexion)
conexion.close()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(df['comprador'], df['num_compras'], color='darkorange')
ax.bar_label(bars, padding=4)
ax.set_xlabel('Número de compras')
ax.set_title('Top 5 compradores con más compras')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

<font color="yellow" size = "4" face = "small fonts">9. Grafique la distribución de precios de los 5 primeros artículos .</font></center><br>

In [ ]:
conexion = sqlite3.connect(SQLITE_PATH)

consulta = """
    SELECT nombreArticulo, precioArticulo
    FROM articulos
    ORDER BY idArticulo ASC
    LIMIT 5
"""
df = pd.read_sql_query(consulta, conexion)
conexion.close()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(df['nombreArticulo'], df['precioArticulo'], color='mediumseagreen')
ax.bar_label(bars, fmt='$%.2f', padding=4)
ax.set_ylabel('Precio (USD)')
ax.set_title('Distribución de precios — 5 primeros artículos')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

<font color="yellow" size = "4" face = "small fonts">10. Grafique los 5 artículos que menos se han vendido.</font></center><br>

In [ ]:
conexion = sqlite3.connect(SQLITE_PATH)

consulta = """
    SELECT a.nombreArticulo, SUM(v.cantidadProductos) AS total_vendido
    FROM ventas v
    JOIN articulos a ON v.idArticulo = a.idArticulo
    GROUP BY v.idArticulo
    ORDER BY total_vendido ASC
    LIMIT 5
"""
df = pd.read_sql_query(consulta, conexion)
conexion.close()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(df['nombreArticulo'], df['total_vendido'], color='tomato')
ax.bar_label(bars, padding=4)
ax.set_xlabel('Unidades vendidas')
ax.set_title('Top 5 artículos menos vendidos')
ax.invert_yaxis()
plt.tight_layout()
plt.show()